# Импорт библиотек

In [2]:
import pandas as pd
import numpy as np

# разбиение на train и test
from sklearn.model_selection import train_test_split

# построение пайплайна
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# преобразование данных
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# модель
from sklearn.linear_model import LogisticRegression

## 1.Загрузка данных

In [9]:
csv= pd.read_csv('Telco-Customer-Churn.csv')

In [11]:
csv.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

**Увидели благодаря csv.info(), что TotalCharges имеет тип object. Ищим невидимые символы в столбце TotalCharges. Превращаем ошибки в NaN. Параметр errors='coerce' заменит значения, которые невозможно преобразовать в число, на NaN.**

In [12]:
csv['TotalCharges'] = pd.to_numeric(csv['TotalCharges'],errors='coerce')

**Проверяем результат:**

In [13]:
print(csv['TotalCharges'].dtype)

float64


**Тип данных изменился на float64 Узнаем количество строк с аномалиями Метод isna() найдет все получившиеся пропуски, а sum() их сложит**

In [14]:
pd.to_numeric(csv['TotalCharges'], errors='coerce').isna().sum()

np.int64(11)

**Выводим на экран аномалии:**

In [15]:
csv[pd.to_numeric(csv['TotalCharges'], errors='coerce').isna()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


## Пункт2: Базовая очистка данных 

**Удаляем customerID.Делаем его индексом датафрейма**

In [16]:
csv.set_index('customerID', drop=True, inplace=True)

**Проверяем значения в customerID**

In [17]:
csv.head(10)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
customerID,,,,,,,,,,,,,,,,,,,,
7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes
1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,No
6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.90,No
7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes


**Преобразуем целевую переменную Churn из текста («Yes»/«No») в бинарные (1/0)**

In [18]:
map_dict = {'Yes': 1, 'No': 0}
csv['Churn'] = csv['Churn'].map(map_dict)

**Проверяем преобразование**

In [19]:
print(csv['Churn'].head())

customerID
7590-VHVEG    0
5575-GNVDE    0
3668-QPYBK    1
7795-CFOCW    0
9237-HQITU    1
Name: Churn, dtype: int64


**Так как в столбце TotalCharges пробелы мы уже преобразовали ранее в NaN переходим к заполнению пропусков обоснованным значением (нулем или медианой). Логичнее выбрать заполнение нулем, так как у клиентов с TotalCharges NaN стаж везде заполнен 0 это означает, что кол-во стажа еще недостаточно для того чтобы набралась итоговая сумма.**

In [20]:
csv['TotalCharges'] = csv['TotalCharges'].fillna(0)

**Проверяем замену:**

In [21]:
print(csv['TotalCharges'].isna().sum())

0


## 3.Разделяем данные на общие, валидационные и тестовые выборки

In [22]:
# Разделение на признаки и целевую переменную
X = csv.drop('Churn', axis=1)
y = csv['Churn']

# Сначала выделяем 60% для обучения
# Оставшиеся 40% будут разделены на validation и test
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=42,
    stratify=y
)
# Делим оставшиеся 40% пополам:
# 20% — validation и 20% — test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

**Прорверяем пропорции выборок:**

In [24]:
print(f'Train: {len(X_train) / len(X):.2%}')
print(f'Validation: {len(X_valid) / len(X):.2%}')
print(f'Test: {len(X_test) / len(X):.2%}')

Train: 59.99%
Validation: 20.01%
Test: 20.01%


## 4.Создаем объект ColumnTransformer внутри Pipeline

In [25]:
# Создаем пайплайн
# Определяем числовые и категориальные признаки
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_features =['gender', 'SeniorCitizen','Partner', 'Dependents','PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
               'TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']
# Пайплайн для числовых признаков
# Обработка числовых признаков:
# 1. Заполнение пропусков медианой
# 2. Масштабирование
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
# Обработка категориальных признаков:
# 1. Заполнение пропусков самым частым значением (strategy='most_frequent')
# 2. One-Hot кодирование ('onehot', OneHotEncoder(handle_unknown='ignore'))
# handle_unknown='ignore' нужен, чтобы модель не выдавала ошибку, если в новых данных встретится категория, которой не было в обучающей выборке
cat_pipeline = Pipeline ([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


# Объединяем все через ColumnTransformer
preprocessor = ColumnTransformer([
('num', num_pipeline, num_features),
('cat', cat_pipeline, cat_features)
])

# Финальный Pipeline
pipeline = Pipeline([
('preprocessing', preprocessor),
('model', LogisticRegression())
])